# SHL Assessment Recommendation Engine - Debug & Testing Notebook
This notebook walks through the full pipeline step-by-step:
1. Inspect scraped data (FAISS stored documents)
2. See how embeddings/text representations look
3. Query → retrieval → see which chunks FAISS returns
4. Re-ranking → final answer
5. Evaluate on training data with Recall@10

In [1]:
import json
import numpy as np
import pandas as pd
from collections import defaultdict
import openpyxl

import config
from embeddings import load_index, build_text_representation, get_embeddings, embed_query, get_embeddings_model

print(f"OpenAI API Key loaded: {config.OPENAI_API_KEY[:10]}...")

c:\Users\A8000\Downloads\Ashish_Genai_project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenAI API Key loaded: sk-proj-4u...


## 1. Inspect Scraped Data (What's stored in FAISS)

In [2]:
# Load the raw scraped assessments
with open(config.ASSESSMENTS_FILE, 'r', encoding='utf-8') as f:
    assessments = json.load(f)

print(f"Total assessments scraped: {len(assessments)}")
print(f"Individual: {sum(1 for a in assessments if a.get('category') == 'Individual Test Solutions')}")
print(f"Pre-packaged: {sum(1 for a in assessments if a.get('category') == 'Pre-packaged Job Solutions')}")
print(f"With description: {sum(1 for a in assessments if a.get('description'))}")
print(f"With duration: {sum(1 for a in assessments if a.get('duration_minutes'))}")

Total assessments scraped: 518
Individual: 377
Pre-packaged: 141
With description: 518
With duration: 432


In [3]:
# Show 5 sample assessments as a table
sample_df = pd.DataFrame(assessments[:5])
sample_df[['name', 'url', 'description', 'test_types', 'duration_minutes', 'remote_testing', 'category']]

,name,url,description,test_types,duration_minutes,remote_testing,category
0,Global Skills Development Report,https://www.shl.com/products/product-catalog/v...,This report is designed to be given to individ...,"[Ability & Aptitude, Assessment Exercises, Bio...",NaN,True,Individual Test Solutions
1,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,The.NET Framework 4.5 test measures knowledge ...,[Knowledge & Skills],30.0,True,Individual Test Solutions
2,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,Multi-choice test that measures the knowledge ...,[Knowledge & Skills],17.0,True,Individual Test Solutions
3,.NET MVVM (New),https://www.shl.com/products/product-catalog/v...,Multi-choice test that measures the knowledge ...,[Knowledge & Skills],5.0,True,Individual Test Solutions
4,.NET WCF (New),https://www.shl.com/products/product-catalog/v...,Multi-choice test that measures the knowledge ...,[Knowledge & Skills],11.0,True,Individual Test Solutions


## 2. Inspect FAISS Index & Text Representations (Chunks)

In [4]:
# Load the FAISS index and metadata
index, stored_assessments, stored_texts = load_index()

print(f"FAISS index size: {index.ntotal} vectors")
print(f"Vector dimension: {index.d}")
print(f"Stored assessments: {len(stored_assessments)}")
print(f"Stored text chunks: {len(stored_texts)}")

FAISS index size: 518 vectors
Vector dimension: 3072
Stored assessments: 518
Stored text chunks: 518


In [5]:
# Show what each "chunk" looks like — this is the text that gets embedded
print("=" * 80)
print("SAMPLE CHUNKS (text representations stored in FAISS)")
print("=" * 80)
for i in [0, 50, 100, 200, 300]:
    if i < len(stored_texts):
        print(f"\n--- Chunk {i}: {stored_assessments[i]['name']} ---")
        print(stored_texts[i])
        print(f"   [Char length: {len(stored_texts[i])}]")

SAMPLE CHUNKS (text representations stored in FAISS)

--- Chunk 0: Global Skills Development Report ---
Global Skills Development Report is a Ability & Aptitude, Assessment Exercises, Biodata & Situational Judgement, Competencies, Development & 360, Personality & Behaviour assessment. This report is designed to be given to individuals who have completed the Global Skills Assessment (GSA). With coverage across the Great 8 Domains, this… Suitable for Director, Entry-Level, Executive, General Population, Graduate, Manager, Mid-Professional, Front Line Manager, Supervisor roles.
   [Char length: 477]

--- Chunk 50: Business Communications ---
Business Communications is a Knowledge & Skills assessment. This test measures the candidate's knowledge of communicating in the workplace.  It measures the skills necessary to communicate effectively with coworkers at… Takes 35 minutes to complete.
   [Char length: 249]

--- Chunk 100: Entry Level Customer Serv-Retail & Contact Center ---
Entry Level

In [6]:
# Distribution of chunk lengths
lengths = [len(t) for t in stored_texts]
print(f"Chunk length stats:")
print(f"  Min: {min(lengths)} chars")
print(f"  Max: {max(lengths)} chars")
print(f"  Mean: {np.mean(lengths):.0f} chars")
print(f"  Median: {np.median(lengths):.0f} chars")
print(f"\nAll chunks fit in one embedding (no splitting needed): {max(lengths) < 8000}")

Chunk length stats:
  Min: 102 chars
  Max: 477 chars
  Mean: 328 chars
  Median: 326 chars

All chunks fit in one embedding (no splitting needed): True


## 3. Query → FAISS Retrieval — See Raw Retrieved Chunks

In [7]:
def debug_retrieval(query: str, top_k: int = 10):
    """Show exactly what FAISS returns for a query."""
    # Step 1: Embed the query using LangChain
    q_embedding = embed_query(query)
    q_norm = q_embedding / np.linalg.norm(q_embedding, axis=1, keepdims=True)
    
    # Step 2: FAISS search
    scores, indices = index.search(q_norm, top_k)
    
    print(f"Query: {query[:100]}..." if len(query) > 100 else f"Query: {query}")
    print(f"\n{'Rank':<5} {'Score':<8} {'Name':<45} {'Test Types':<30} {'Duration':<10}")
    print("-" * 100)
    
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
        a = stored_assessments[idx]
        types = ', '.join(a.get('test_types', []))
        dur = str(a.get('duration_minutes', '-'))
        print(f"{rank:<5} {score:<8.4f} {a['name'][:44]:<45} {types[:29]:<30} {dur:<10}")
        results.append({'rank': rank, 'score': score, 'name': a['name'], 'url': a['url']})
    
    return results

In [8]:
# Test query 1: Java developer
results = debug_retrieval("I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes ?", top_k=15)

Query: I am hiring for Java developers who can also collaborate effectively with my business teams. Looking...

Rank  Score    Name                                          Test Types                     Duration  
----------------------------------------------------------------------------------------------------
1     0.5813   Core Java (Entry Level) (New)                 Knowledge & Skills             13        
2     0.5705   Java 8 (New)                                  Knowledge & Skills             18        
3     0.5521   Core Java (Advanced Level) (New)              Knowledge & Skills             13        
4     0.5492   Java Frameworks (New)                         Knowledge & Skills             17        
5     0.5444   Java Web Services (New)                       Knowledge & Skills             8         
6     0.5352   Java Design Patterns (New)                    Knowledge & Skills             5         
7     0.5205   Swing (New)                                   Knowl

In [ ]:
# Test query 2: Data Analyst
results = debug_retrieval("I want to hire new graduates for a sales role in my company, the budget is for about an hour for each test. Give me some options", top_k=15)

In [ ]:
# Test query 3: Bank admin
results = debug_retrieval("ICICI Bank Assistant Admin, Experience 0-2 years, banking administrative role", top_k=15)

In [9]:
# Show the full chunk text that FAISS matched for a query
query = "Content Writer expert in English and SEO"
q_embedding = embed_query(query)
q_norm = q_embedding / np.linalg.norm(q_embedding, axis=1, keepdims=True)
scores, indices = index.search(q_norm, 5)

print(f"Query: {query}\n")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    print(f"--- Retrieved Chunk #{rank} (score={score:.4f}) ---")
    print(f"FULL TEXT: {stored_texts[idx]}")
    print()

Query: Content Writer expert in English and SEO

--- Retrieved Chunk #1 (score=0.3694) ---
FULL TEXT: Search Engine Optimization (New) is a Knowledge & Skills assessment. Multi-choice test that measures the knowledge on the concepts of need of SEO, SEO planning, SEO strategies, SEO software, tools and exchanging links. Suitable for Graduate, Mid-Professional, Professional Individual Contributor roles. Takes 12 minutes to complete.

--- Retrieved Chunk #2 (score=0.3310) ---
FULL TEXT: Written English v1 is a Knowledge & Skills assessment. The Written English test measures knowledge of US English grammar and English reading comprehension. It is designed for those with English as a second language… Suitable for Entry-Level, Front Line Manager, Mid-Professional, Professional Individual Contributor roles. Takes 30 minutes to complete.

--- Retrieved Chunk #3 (score=0.3286) ---
FULL TEXT: WriteX - Email Writing (Sales) (New) is a Biodata & Situational Judgement, Simulations assessment. Open 

## 4. Full Pipeline Debug — Query Analyzer → Retriever → Reranker

In [10]:
from graph import query_analyzer_node, retriever_node, reranker_node, GraphState

def debug_full_pipeline(query: str):
    """Run each pipeline node separately and show intermediate outputs."""
    
    # Initial state
    state = GraphState(
        query=query, search_queries=[], skills=[], job_level="any",
        max_duration=None, domain="",
        candidates=[], recommendations=[]
    )
    
    # --- Node 1: Query Analyzer ---
    print("=" * 80)
    print("NODE 1: QUERY ANALYZER")
    print("=" * 80)
    analyzer_output = query_analyzer_node(state)
    state.update(analyzer_output)
    
    print(f"Generated search queries:")
    for i, sq in enumerate(state['search_queries'], 1):
        print(f"  {i}. {sq}")
    print(f"\nExtracted skills: {state['skills']}")
    print(f"Job level: {state['job_level']}")
    print(f"Max duration: {state['max_duration']}")
    print(f"Domain: {state['domain']}")
    
    # --- Node 2: Retriever ---
    print(f"\n{'=' * 80}")
    print("NODE 2: RETRIEVER (FAISS multi-query search)")
    print("=" * 80)
    retriever_output = retriever_node(state)
    state.update(retriever_output)
    
    print(f"Retrieved {len(state['candidates'])} candidates")
    print(f"\n{'Rank':<5} {'Score':<8} {'Name':<45} {'Test Types':<25}")
    print("-" * 85)
    for i, c in enumerate(state['candidates'][:20], 1):
        types = ', '.join(c.get('test_type', [])[:2])
        print(f"{i:<5} {c['score']:<8.4f} {c['name'][:44]:<45} {types:<25}")
    
    # --- Node 3: Reranker ---
    print(f"\n{'=' * 80}")
    print("NODE 3: RERANKER (LLM re-ranking)")
    print("=" * 80)
    reranker_output = reranker_node(state)
    state.update(reranker_output)
    
    print(f"\nFinal {len(state['recommendations'])} recommendations:")
    print(f"\n{'#':<4} {'Name':<45} {'Test Types':<30} {'Duration':<10} {'URL'}")
    print("-" * 130)
    for i, r in enumerate(state['recommendations'], 1):
        types = ', '.join(r.get('test_type', []))
        dur = str(r.get('duration') or '-')
        print(f"{i:<4} {r['name'][:44]:<45} {types[:29]:<30} {dur:<10} {r['url']}")
    
    return state

In [11]:
# Full pipeline debug for a sample query
state = debug_full_pipeline("I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.")

NODE 1: QUERY ANALYZER
Generated search queries:
  1. Java programming assessment
  2. core Java test
  3. verify numerical ability test
  4. inductive reasoning test
  5. occupational personality questionnaire
  6. interpersonal communication assessment
  7. professional job focused assessment
  8. automata coding simulation test
  9. I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.

Extracted skills: ['Java', 'Java programming', 'collaboration', 'teamwork', 'interpersonal communication']
Job level: any
Max duration: 40
Domain: software development

NODE 2: RETRIEVER (FAISS multi-query search)
Retrieved 60 candidates

Rank  Score    Name                                          Test Types               
-------------------------------------------------------------------------------------
1     0.7424   Occupational Personality Questionnaire OPQ32  Personality & Behaviour  
2  

In [12]:
# Another pipeline debug
state = debug_full_pipeline("I am looking for a COO for my company in China and I want to see if they are culturally a right fit for our company. Suggest me an assessment that they can complete in about an hour")

NODE 1: QUERY ANALYZER
Generated search queries:
  1. COO job focused assessment
  2. executive leadership skills test
  3. situational judgement test for executives
  4. occupational personality questionnaire
  5. cultural fit assessment China
  6. verify inductive reasoning test
  7. verify verbal ability test
  8. I am looking for a COO for my company in China and I want to see if they are culturally a right fit for our company. Suggest me an assessment that they can complete in about an hour

Extracted skills: ['executive leadership', 'operational management', 'strategic thinking', 'cultural fit', 'personality', 'verbal reasoning', 'inductive reasoning']
Job level: any
Max duration: 60
Domain: executive management

NODE 2: RETRIEVER (FAISS multi-query search)
Retrieved 60 candidates

Rank  Score    Name                                          Test Types               
-------------------------------------------------------------------------------------
1     0.7424   Occupational 

## 5. Evaluate on Training Data — Recall@10 per query

In [13]:
from evaluate import normalize_url, load_train_set, compute_recall_at_k
from graph import recommend

# Load training data
dataset_path = 'input/Gen_AI Dataset.xlsx'
train_data = load_train_set(dataset_path)

print(f"Training queries: {len(train_data)}")
print(f"Total labeled URLs: {sum(len(v) for v in train_data.values())}")
print()
for i, (q, urls) in enumerate(train_data.items(), 1):
    print(f"  Q{i} ({len(urls)} labels): {q[:80]}...")

Training queries: 10
Total labeled URLs: 65

  Q1 (5 labels): I am hiring for Java developers who can also collaborate effectively with my bus...
  Q2 (9 labels): I want to hire new graduates for a sales role in my company, the budget is for a...
  Q3 (6 labels): I am looking for a COO for my company in China and I want to see if they are cul...
  Q4 (5 labels): KEY RESPONSIBITILES:

Manage the sound-scape of the station through appropriate ...
  Q5 (5 labels): Content Writer required, expert in English and SEO....
  Q6 (9 labels): Find me 1 hour long assesment for the below job at SHL
Job Description

 Join a ...
  Q7 (6 labels): ICICI Bank Assistant Admin, Experience required 0-2 years, test should be 30-40 ...
  Q8 (5 labels): We're looking for a Marketing Manager who can drive Recro’s brand positioning, c...
  Q9 (5 labels): Based on the JD below recommend me assessment for the Consultant position in my ...
  Q10 (10 labels): I want to hire a Senior Data Analyst with 5 years of exp

In [14]:
# Run evaluation on all training queries
results_table = []

for i, (query, relevant_urls) in enumerate(train_data.items(), 1):
    recommendations = recommend(query)
    rec_urls = [r['url'] for r in recommendations]
    
    recall = compute_recall_at_k(rec_urls, relevant_urls, k=10)
    
    # Count hits/misses
    rec_normalized = [normalize_url(u) for u in rec_urls]
    hits = []
    misses = []
    for url in relevant_urls:
        name = url.split('/view/')[-1].rstrip('/')
        if normalize_url(url) in rec_normalized:
            hits.append(name)
        else:
            misses.append(name)
    
    results_table.append({
        'Query': f"Q{i}: {query[:60]}...",
        'Relevant': len(relevant_urls),
        'Hits': len(hits),
        'Misses': len(misses),
        'Recall@10': f"{recall:.2f}",
        'Hit Names': ', '.join(hits[:3]),
        'Miss Names': ', '.join(misses[:3]),
    })
    print(f"Q{i}: Recall@10={recall:.2f} | Hits={len(hits)}/{len(relevant_urls)}")

mean_recall = np.mean([float(r['Recall@10']) for r in results_table])
print(f"\n>>> Mean Recall@10 = {mean_recall:.4f} <<<")

Q1: Recall@10=1.00 | Hits=5/5
Q2: Recall@10=0.33 | Hits=3/9
Q3: Recall@10=0.50 | Hits=3/6
Q4: Recall@10=0.60 | Hits=3/5
Q5: Recall@10=0.80 | Hits=4/5
Q6: Recall@10=0.56 | Hits=5/9
Q7: Recall@10=0.50 | Hits=3/6
Q8: Recall@10=0.40 | Hits=2/5
Q9: Recall@10=0.60 | Hits=3/5
Q10: Recall@10=0.30 | Hits=3/10

>>> Mean Recall@10 = 0.5590 <<<


In [ ]:
# Show results as a DataFrame
results_df = pd.DataFrame(results_table)
results_df

In [ ]:
# Deep dive into the worst performing query — see what went wrong
worst_idx = np.argmin([float(r['Recall@10']) for r in results_table])
worst_query = list(train_data.keys())[worst_idx]
worst_urls = train_data[worst_query]

print(f"WORST QUERY (Q{worst_idx+1}): {worst_query[:100]}...")
print(f"\nExpected URLs ({len(worst_urls)}):")
for u in worst_urls:
    name = u.split('/view/')[-1].rstrip('/')
    print(f"  - {name}")

print(f"\n--- Running full debug pipeline ---")
state = debug_full_pipeline(worst_query)

## 6. Embedding Similarity Analysis

In [ ]:
# Check: are the expected assessments even close in embedding space?
# For the worst query, embed it and check distance to expected assessments
from evaluate import normalize_url as norm_url_check

q_emb = embed_query(worst_query[:500])
q_norm = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)

print("Cosine similarity of EXPECTED assessments to the query:")
print(f"{'Assessment':<50} {'Similarity':<10} {'In Top-40?'}")
print("-" * 75)

# Get top-40 indices for comparison
_, top40_indices = index.search(q_norm, 40)
top40_set = set(top40_indices[0].tolist())

for expected_url in worst_urls:
    # Find this URL in our assessments
    found = False
    for idx, a in enumerate(stored_assessments):
        if norm_url_check(a['url']) == norm_url_check(expected_url):
            # Reconstruct the embedding vector from index
            vec = index.reconstruct(idx).reshape(1, -1)
            sim = float(np.dot(q_norm[0], vec[0]))
            in_top40 = "YES" if idx in top40_set else "NO"
            name = expected_url.split('/view/')[-1].rstrip('/')
            print(f"{name:<50} {sim:<10.4f} {in_top40}")
            found = True
            break
    if not found:
        name = expected_url.split('/view/')[-1].rstrip('/')
        print(f"{name:<50} {'NOT IN INDEX':<10} {'N/A'}")